In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision.datasets import CIFAR10

In [3]:
import torch
from torchvision import transforms
from torchvision.datasets import CIFAR10

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

trainset = CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

testset = CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("Training images:", len(trainset))
print("Test images:", len(testset))

Training images: 50000
Test images: 10000


In [4]:
train_loader=DataLoader(trainset,batch_size=64,shuffle=True)
test_loader=DataLoader(testset,batch_size=64)

## Build CNN  Model

In [12]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
            

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1), #Input Channels,no of total filters,kernal size,padding
            nn.ReLU(),
            nn.MaxPool2d(2,2), #Kernal Size Stride

            nn.Conv2d(32,64,kernel_size=3,padding=1), #Input Channels,no of total filters,kernal size,padding
            nn.ReLU(),
            nn.MaxPool2d(2,2), #Kernal Size Stride

            nn.Conv2d(64 , 128,kernel_size=3,padding=1), #Input Channels,no of total filters,kernal size,padding
            nn.ReLU(),
            nn.MaxPool2d(2 , 2) #Kernal Size Stride
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(2048,256), #after pooling it decreases width and height by half wo get 4*4*128
            nn.ReLU(),
            nn.Linear(256 , 10) # we are using multi classification so we have 10 outputs on which softmax will be applied default
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1) #Flattening
        x =self.fc_layers(x)
        return x    

In [13]:
model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())


In [23]:
#Train the CNN
epochs=10

for epoch in range(epochs):
    training_epoch_loss=0.0

    for image,label in train_loader:
        optimizer.zero_grad()

        output=model.forward(image)
        loss=criterion(output,label)
        loss.backward()
        optimizer.step()


        training_epoch_loss += loss.item()

    print("epoch : ",(epoch+1) , " / " ,epochs,"--> loss :",training_epoch_loss/len(train_loader))
        
    
    
    


epoch :  1  /  10 --> loss : 0.14421887179869977
epoch :  2  /  10 --> loss : 0.12122642727868865
epoch :  3  /  10 --> loss : 0.1039709964233553
epoch :  4  /  10 --> loss : 0.10034377477429521
epoch :  5  /  10 --> loss : 0.0916029780184912
epoch :  6  /  10 --> loss : 0.08011502059726783
epoch :  7  /  10 --> loss : 0.0811097634523926
epoch :  8  /  10 --> loss : 0.0771125059537923
epoch :  9  /  10 --> loss : 0.07377943496161457
epoch :  10  /  10 --> loss : 0.06713459780737471


In [22]:
#Evaluation of CNN
correct=0
total=0
model.eval()

with torch.no_grad():
    for image,label in test_loader:
        output=model(image)
        _, predict=torch.max(output,1)

        correct += (predict==label).sum().item()
        total += label.size(0)
    print("Accuracy :",correct/total * 100 ," %")    

Accuracy : 75.94  %
